In [0]:
print("hello, my pipeline starts here")

hello, my pipeline starts here


1.1 Store API key

In [0]:
dbutils.widgets.text("api_key", "", "Enter your Alpha Vantage API Key")

1.2 Read the key 

In [0]:
api_key = dbutils.widgets.get("api_key")
print("Key loaded! Length:", len(api_key))

Key loaded! Length: 16


1.3 Make the actual API call

In [0]:
import requests

url = "https://www.alphavantage.co/query"
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": "AAPL",
    "apikey": api_key,
    "outputsize": "compact"
}

response = requests.get(url, params=params)
print("Status code:", response.status_code)

Status code: 200


1.4 Look at the actual data

In [0]:
data = response.json()
print(data.keys())

dict_keys(['Meta Data', 'Time Series (Daily)'])


In [0]:
print(data["Meta Data"])

{'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'AAPL', '3. Last Refreshed': '2026-07-17', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}


2 Save the raw data (Bronze layer) 

2.1 place to store 

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS stock_project;
CREATE SCHEMA IF NOT EXISTS stock_project.pipeline;

2.2 Turn the JSON into a table row

In [0]:
import json
from datetime import datetime
from pyspark.sql import Row

bronze_rows = []
for ticker, result in all_data.items():
    bronze_rows.append(Row(
        ticker=ticker,
        raw_json=json.dumps(result),
        ingestion_timestamp=datetime.now(),
        source="alphavantage_time_series_daily"
    ))

bronze_df = spark.createDataFrame(bronze_rows)
bronze_df.show()

+------+--------------------+--------------------+--------------------+
|ticker|            raw_json| ingestion_timestamp|              source|
+------+--------------------+--------------------+--------------------+
|  AAPL|{"Meta Data": {"1...|2026-07-18 22:36:...|alphavantage_time...|
|  MSFT|{"Meta Data": {"1...|2026-07-18 22:36:...|alphavantage_time...|
| GOOGL|{"Meta Data": {"1...|2026-07-18 22:36:...|alphavantage_time...|
+------+--------------------+--------------------+--------------------+



2.3 Save permamnently

In [0]:
bronze_df.write.format("delta").mode("append") \
    .saveAsTable("stock_project.pipeline.bronze_stock_raw")

2.4 Check it saved correctly

In [0]:
%sql
SELECT * FROM stock_project.pipeline.bronze_stock_raw;

ticker raw_json ingestion_timestamp source AAPL {"Meta Data": {"1. Information": "Daily Prices (open, high, low, close) and Volumes", "2. Symbol": "AAPL", "3. Last Refreshed": "2026-07-17", "4. Output Size": "Compact", "5. Time Zone": "US/Eastern"}, "Time Series (Daily)": {"2026-07-17": {"1. open": "331.9800", "2. high": "334.9900", "3. low": "329.0006", "4. close": "333.7400", "5. volume": "63407059"}, "2026-07-16": {"1. open": "328.0050", "2. high": "334.6800", "3. low": "326.7900", "4. close": "333.2600", "5. volume": "62970617"}, "2026-07-15": {"1. open": "317.6150", "2. high": "328.7300", "3. low": "317.3200", "4. close": "327.5000", "5. volume": "60957644"}, "2026-07-14": {"1. open": "313.7600", "2. high": "316.1900", "3. low": "311.9100", "4. close": "314.8600", "5. volume": "36336829"}, "2026-07-13": {"1. open": "317.0150", "2. high": "323.4500", "3. low": "315.7800", "4. close": "317.3100", "5. volume": "43257804"}, "2026-07-10": {"1. open": "314.7200", "2. high": "316.9100", "3. low": "312.1700", "4. close": "315.3200", "5. volume": "34132321"}, "2026-07-09": {"1. open": "310.5100", "2. high": "316.5300", "3. low": "308.1600", "4. close": "316.2200", "5. volume": "48124490"}, "2026-07-08": {"1. open": "311.9100", "2. high": "314.8200", "3. low": "307.0500", "4. close": "313.3900", "5. volume": "41323480"}, "2026-07-07": {"1. open": "315.2900", "2. high": "315.4800", "3. low": "310.1500", "4. close": "310.6600", "5. volume": "42490002"}, "2026-07-06": {"1. open": "307.3600", "2. high": "314.2000", "3. low": "307.0000", "4. close": "312.6600", "5. volume": "53589977"}, "2026-07-02": {"1. open": "294.1200", "2. high": "309.4200", "3. low": "293.6800", "4. close": "308.6300", "5. volume": "75400626"}, "2026-07-01": {"1. open": "293.4400", "2. high": "296.5900", "3. low": "289.1950", "4. close": "294.3800", "5. volume": "50164232"}, "2026-06-30": {"1. open": "281.1700", "2. high": "289.9400", "3. low": "280.6950", "4. close": "289.3600", "5. volume": "65100155"}, "2026-06-29": {"1. open": "286.7300", "2. high": "288.3697", "3. low": "279.8500", "4. close": "281.7400", "5. volume": "66427002"}, "2026-06-26": {"1. open": "275.0000", "2. high": "285.9500", "3. low": "274.2100", "4. close": "283.7800", "5. volume": "261775450"}, "2026-06-25": {"1. open": "287.4000", "2. high": "288.8000", "3. low": "273.7500", "4. close": "275.1500", "5. volume": "107253659"}, "2026-06-24": {"1. open": "295.3550", "2. high": "299.7000", "3. low": "292.9400", "4. close": "293.0800", "5. volume": "53081859"}, "2026-06-23": {"1. open": "297.5380", "2. high": "301.6400", "3. low": "294.1800", "4. close": "294.3000", "5. volume": "52010929"}, "2026-06-22": {"1. open": "297.3100", "2. high": "302.4200", "3. low": "296.7600", "4. close": "297.0100", "5. volume": "44879914"}, "2026-06-18": {"1. open": "298.1100", "2. high": "300.5700", "3. low": "295.6200", "4. close": "298.0100", "5. volume": "85962201"}, "2026-06-17": {"1. open": "300.8450", "2. high": "302.0700", "3. low": "294.3600", "4. close": "295.9500", "5. volume": "42745060"}, "2026-06-16": {"1. open": "295.2450", "2. high": "300.4800", "3. low": "293.9700", "4. close": "299.2400", "5. volume": "39874404"}, "2026-06-15": {"1. open": "294.1200", "2. high": "297.7800", "3. low": "291.7000", "4. close": "296.4200", "5. volume": "45732573"}, "2026-06-12": {"1. open": "296.0300", "2. high": "297.1400", "3. low": "289.6200", "4. close": "291.1300", "5. volume": "38784789"}, "2026-06-11": {"1. open": "293.7200", "2. high": "297.0000", "3. low": "289.5900", "4. close": "295.6300", "5. volume": "42572497"}, "2026-06-10": {"1. open": "290.7400", "2. high": "294.7500", "3. low": "287.3800", "4. close": "291.5800", "5. volume": "52793266"}, "2026-06-09": {"1. open": "300.2750", "2. high": "300.7500", "3. low": "287.7800", "4. close": "290.5500", "5. volume": "70108847"}, "2026-06-08": {"1. open": "308.7390", "2. high": "317.4000", "3. low": "301.1700", "4. close": "301.5400", "5. volume": "77949082"}, 

3 Clean the data (Silver layer)

3.1 Understand the shape of the data

In [0]:
time_series = data["Time Series (Daily)"]
first_date = list(time_series.keys())[0]
print(first_date)
print(time_series[first_date])

2026-07-17
{'1. open': '331.9800', '2. high': '334.9900', '3. low': '329.0006', '4. close': '333.7400', '5. volume': '63407059'}


3.2 Loop through every day and build clean rows

In [0]:
from pyspark.sql import Row

silver_rows = []
for date_str, values in time_series.items():
    silver_rows.append(Row(
        ticker="AAPL",
        trade_date=date_str,
        open_price=float(values["1. open"]),
        high_price=float(values["2. high"]),
        low_price=float(values["3. low"]),
        close_price=float(values["4. close"]),
        volume=int(values["5. volume"])
    ))

print("Total days processed:", len(silver_rows))

Total days processed: 100


3.3 Turn it into a table and save it

In [0]:
from pyspark.sql.functions import col, to_date

silver_df = spark.createDataFrame(silver_rows) \
    .withColumn("trade_date", to_date(col("trade_date")))

silver_df.write.format("delta").mode("overwrite") \
    .saveAsTable("stock_project.pipeline.silver_stock_prices")

silver_df.show(10)

+------+----------+----------+----------+---------+-----------+--------+
|ticker|trade_date|open_price|high_price|low_price|close_price|  volume|
+------+----------+----------+----------+---------+-----------+--------+
|  AAPL|2026-07-17|    331.98|    334.99| 329.0006|     333.74|63407059|
|  AAPL|2026-07-16|   328.005|    334.68|   326.79|     333.26|62970617|
|  AAPL|2026-07-15|   317.615|    328.73|   317.32|      327.5|60957644|
|  AAPL|2026-07-14|    313.76|    316.19|   311.91|     314.86|36336829|
|  AAPL|2026-07-13|   317.015|    323.45|   315.78|     317.31|43257804|
|  AAPL|2026-07-10|    314.72|    316.91|   312.17|     315.32|34132321|
|  AAPL|2026-07-09|    310.51|    316.53|   308.16|     316.22|48124490|
|  AAPL|2026-07-08|    311.91|    314.82|   307.05|     313.39|41323480|
|  AAPL|2026-07-07|    315.29|    315.48|   310.15|     310.66|42490002|
|  AAPL|2026-07-06|    307.36|     314.2|    307.0|     312.66|53589977|
+------+----------+----------+----------+---------+

4  Calculate real insights (Gold layer)

4.1 Calculate a 7-day moving average

In [0]:
%sql
CREATE OR REPLACE TABLE stock_project.pipeline.gold_stock_metrics AS
SELECT
  ticker,
  trade_date,
  close_price,
  ROUND(
    AVG(close_price) OVER (
      PARTITION BY ticker ORDER BY trade_date
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2
  ) AS moving_avg_7day
FROM stock_project.pipeline.silver_stock_prices;

num_affected_rows,num_inserted_rows


In [0]:
gold_df = spark.sql("SELECT * FROM stock_project.pipeline.gold_stock_metrics")
gold_df.show(5)

+------+----------+-----------+---------------+
|ticker|trade_date|close_price|moving_avg_7day|
+------+----------+-----------+---------------+
|  AAPL|2026-02-24|     272.14|         272.14|
|  AAPL|2026-02-25|     274.23|         273.19|
|  AAPL|2026-02-26|     272.95|         273.11|
|  AAPL|2026-02-27|     264.18|         270.88|
|  AAPL|2026-03-02|     264.72|         269.64|
+------+----------+-----------+---------------+
only showing top 5 rows


In [0]:
dbutils.widgets.text("sf_user", "", "Snowflake Username")
dbutils.widgets.text("sf_password", "", "Snowflake Password")
dbutils.widgets.text("sf_account", "dtzhadz-va37760", "Snowflake Account")

In [0]:
sf_user = dbutils.widgets.get("sf_user")
sf_password = dbutils.widgets.get("sf_password")
sf_account = dbutils.widgets.get("sf_account")

In [0]:
sfOptions = {
    "host": f"{sf_account}.snowflakecomputing.com",
    "port": "443",
    "sfUser": sf_user,
    "sfPassword": sf_password,
    "sfDatabase": "STOCK_MARKET_DB",
    "sfSchema": "GOLD",
    "sfWarehouse": "STOCK_WH"
}

gold_df.write \
    .format("net.snowflake.spark.snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "STOCK_MOVING_AVG") \
    .mode("overwrite") \
    .save()

4.2 See the result

In [0]:
%sql
SELECT * FROM stock_project.pipeline.gold_stock_metrics
ORDER BY trade_date DESC;

5 scaling from 1 stock to multiple

5.1 Pull data for 3 tickers

In [0]:
tickers = ["AAPL", "MSFT", "GOOGLE"]
all_data = {}

for ticker in tickers:
    params["symbol"] = ticker
    response = requests.get(url, params=params)
    if response.status_code == 200:
        all_data[ticker] = response.json()
        print(f"{ticker}: success")
    else:
        print(f"{ticker}: failed with {response.status_code}")

AAPL: success
MSFT: success
GOOGLE: success


5.2 Update flattening loop to handle all 3 tickers

In [0]:
for ticker, ticker_data in all_data.items():
    print(ticker, "->", list(ticker_data.keys()))

AAPL -> ['Meta Data', 'Time Series (Daily)']
MSFT -> ['Information']
GOOGLE -> ['Information']


In [0]:
import time

tickers = ["AAPL", "MSFT", "GOOGL"]
all_data = {}

for ticker in tickers:
    params["symbol"] = ticker
    response = requests.get(url, params=params)
    result = response.json()

    if "Time Series (Daily)" in result:
        all_data[ticker] = result
        print(f"{ticker}: success")
    else:
        print(f"{ticker}: failed - {result}")

    time.sleep(15)  # wait 15 seconds before the next request

AAPL: success
MSFT: success
GOOGL: success


In [0]:
from pyspark.sql import Row

silver_rows = []
for ticker, ticker_data in all_data.items():
    time_series = ticker_data["Time Series (Daily)"]
    for date_str, values in time_series.items():
        silver_rows.append(Row(
            ticker=ticker,
            trade_date=date_str,
            open_price=float(values["1. open"]),
            high_price=float(values["2. high"]),
            low_price=float(values["3. low"]),
            close_price=float(values["4. close"]),
            volume=int(values["5. volume"])
        ))

print("Total rows:", len(silver_rows))

Total rows: 300


In [0]:
from pyspark.sql.functions import col, to_date

silver_df = spark.createDataFrame(silver_rows) \
    .withColumn("trade_date", to_date(col("trade_date")))

silver_df.write.format("delta").mode("overwrite") \
    .saveAsTable("stock_project.pipeline.silver_stock_prices")

silver_df.groupBy("ticker").count().show()

In [0]:
%sql
CREATE OR REPLACE TABLE stock_project.pipeline.gold_stock_metrics AS
SELECT
  ticker,
  trade_date,
  close_price,
  ROUND(
    AVG(close_price) OVER (
      PARTITION BY ticker ORDER BY trade_date
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2
  ) AS moving_avg_7day
FROM stock_project.pipeline.silver_stock_prices;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT ticker, COUNT(*) AS days
FROM stock_project.pipeline.gold_stock_metrics
GROUP BY ticker;

In [0]:
%sql
SELECT * FROM stock_project.pipeline.gold_stock_metrics
ORDER BY trade_date ASC;

In [0]:
%sql
SELECT TICKER, COUNT(*) FROM stock_project.pipeline.gold_stock_metrics GROUP BY TICKER;

In [0]:
%sql
SELECT TICKER, COUNT(*) FROM stock_project.pipeline.silver_stock_prices GROUP BY TICKER;

In [0]:
%sql
SELECT ticker, COUNT(*) FROM stock_project.pipeline.bronze_stock_prices GROUP BY ticker;

In [0]:
%sql
DELETE FROM stock_project.pipeline.bronze_stock_raw;

num_affected_rows
3


In [0]:
%sql
SELECT ticker, COUNT(*) FROM stock_project.pipeline.gold_stock_metrics GROUP BY ticker;

ticker,COUNT(*)
AAPL,100
